<a href="https://colab.research.google.com/github/bibidhSubedi0/NepalLawFT/blob/main/notebooks/Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset

In [ ]:
judge = LangchainLLMWrapper(
    ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0,
        api_key=os.environ["GROQ_API_KEY"]
    )
)

In [ ]:
embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
)

In [ ]:
faithfulness.llm            = judge
answer_relevancy.llm        = judge
answer_relevancy.embeddings = embeddings
context_precision.llm       = judge

In [ ]:
test_data = [
    {
        "question": "What does Article 18 of the Nepal constitution say about equality?",
        "answer": "Article 18 guarantees equality before the law for all citizens and prohibits discrimination based on religion, caste, sex, or origin.",
        "contexts": ["Article 18 of the Constitution of Nepal guarantees the right to equality. All citizens shall be equal before the law. No person shall be discriminated against on grounds of religion, race, caste, tribe, sex, origin, language or ideological conviction."],
        "ground_truth": "Article 18 guarantees equality before law and prohibits discrimination on grounds of religion, caste, sex, or origin."
    },
    {
        "question": "What does Article 18 of the Nepal constitution say about equality?",
        "answer": "Article 22 establishes a supreme equality commission with enforcement powers that can override local courts in discrimination cases.",
        "contexts": ["Article 18 of the Constitution of Nepal guarantees the right to equality. All citizens shall be equal before the law. No person shall be discriminated against on grounds of religion, race, caste, tribe, sex, origin, language or ideological conviction."],
        "ground_truth": "Article 18 guarantees equality before law and prohibits discrimination on grounds of religion, caste, sex, or origin."
    },
]

In [ ]:

dataset = Dataset.from_list(test_data)

scores = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
)

In [ ]:
print(scores)